# Merge Stage 1 -- Build Panel: Macro Monthly

## Purpose
Combines two monthly macro-level cleaned datasets into a single panel keyed on `date`, with publication lags applied to prevent look-ahead bias. This produces the macro-level monthly panel (Panel D).

## Sources (All Cleaned)
- `Data/Data_Collection/Cleaned/02_FRED/fred_monthly_macro_clean.parquet` -- 83 macro indicators (industrial production, unemployment, housing, money supply, inflation, consumer spending, etc.)
- `Data/Data_Collection/Cleaned/09_Macro_Daily_Monthly_WRDS/macro_monthly_clean.parquet` -- 22 factors (Treasury bond returns/indices across the maturity curve, T-bill returns, CPI return/index, Pastor-Stambaugh liquidity factor)

## Key Design Decisions

### Publication Lag Policy
- **FRED monthly factors: 2-month lag.** Most macro releases (CPI, employment, industrial production, housing) have 1--3 month publication delays. A blanket 2-month shift is conservative: January's data gets a date of March, meaning it enters the model in March. This may lose ~1 month of timeliness on fast-release series (e.g., employment, published first Friday of the following month), but guarantees zero look-ahead on slow-release series. For a dissertation where look-ahead avoidance is paramount, this tradeoff is acceptable.
- **WRDS Treasury returns/indices and Pastor-Stambaugh: no lag.** These are computed from market prices known at month-end.
- **WRDS CPI (`cpiret`, `cpiind`): 1-month lag.** CPI for month M is published mid-month M+1 by the BLS. January CPI becomes available in February.

### Other Decisions
- Monthly dates merged on year-month (`_ym`) period keys to handle calendar month-end vs last-trading-day mismatches.
- No winsorisation, no z-standardisation -- deferred to later stages.

## Merge Logic

### Step 1: Load FRED Monthly and Apply Publication Lag
FRED monthly data is loaded and the date column is shifted forward by 2 months (`+ pd.DateOffset(months=2)`). January 2020 data gets a date of March 2020. Rows pushed beyond 2024-12-31 are dropped (the last 2 months of reference data fall outside the sample after the shift). A `_ym` key is added for merging.

### Step 2: Load WRDS Macro Monthly
WRDS macro monthly data is loaded and split into two groups:
- **Market-price columns** (Treasury bond returns/indices, T-bill returns, Pastor-Stambaugh liquidity): no lag applied
- **CPI columns** (`cpiret`, `cpiind`): shifted forward by 1 month. January CPI becomes available in February.

Both groups receive `_ym` keys for merging.

### Step 3: Establish Monthly Spine and Merge
A complete monthly calendar from 2004-01-31 to 2024-12-31 (calendar month-ends) is created as the spine. Three sequential left merges on `_ym`:
1. FRED monthly (already 2-month lagged)
2. WRDS market-price columns (no lag)
3. WRDS CPI columns (1-month lagged)

Column name conflicts between FRED and WRDS are resolved with a `wrds_` prefix. Each join is verified to produce no row explosion.

### Step 4: Column Inventory
All factors categorised by source and lag treatment: FRED monthly (2-month lag), WRDS market (no lag), WRDS CPI (1-month lag).

### Step 5: Validation
- No duplicate dates
- Row count (252 months = 21 years x 12), date range, total columns
- Rows per year (should all be 12)
- NaN summary by source
- **Publication lag verification:** confirms FRED factors are NaN for the first ~2 months (the lag pushes the earliest reference data forward), CPI factors NaN for the first ~1 month, and WRDS market-price factors start from January 2004 (no lag)
- Columns with remaining NaN listed (top 15)
- Date frequency check (calendar month-end)

## Output
`Data/Data_Collection/Final/Stage_1/panel_macro_monthly.parquet` -- keyed on `date` (calendar month-end), containing all monthly macro factors with publication lags applied. Factor breakdown: 83 FRED monthly (2-month lag), ~20 WRDS market (no lag), ~2 WRDS CPI (1-month lag).

In [1]:
# %% [markdown]
# # Merge Pipeline — Notebook 04: Build Panel D (Macro Monthly)
#
# Combines two monthly macro-level datasets into a single panel keyed on
# (date), with publication lag applied to prevent look-ahead bias.
#
# Sources:
#   - FRED monthly — 83 macro indicators (industrial production, unemployment,
#     housing, money supply, etc.)
#   - WRDS macro monthly — 22 factors (Treasury bond returns/indices across
#     maturity curve, T-bill returns, CPI return/index, Pastor-Stambaugh
#     liquidity factor)
#
# KEY DESIGN DECISIONS:
#   - FRED monthly factors are LAGGED by 2 months. Most macro releases
#     (CPI, employment, industrial production) have 1-3 month publication
#     delays. A blanket 2-month lag is conservative: we may lose 1 month of
#     timeliness on fast-release series, but we guarantee zero look-ahead
#     on slow-release series. January's data becomes available to the model
#     in March.
#   - WRDS Treasury returns/indices and Pastor-Stambaugh: NO lag. These are
#     computed from market prices known at month-end.
#   - WRDS CPI (cpiret, cpiind): LAGGED by 1 month. CPI for month M is
#     published mid-month M+1 by the BLS. January CPI available in February.
#   - Monthly dates merged on year-month (_ym) to handle calendar month-end
#     vs last-trading-day mismatches.
#   - No winsorisation, no z-standardisation — those happen in Step 3.
#
# Output: Data/Data_Collection/Final/Stage_1/panel_macro_monthly.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
CLEANED = Path('../../../Data/Data_Collection/Cleaned')
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_1')
OUT_DIR.mkdir(parents=True, exist_ok=True)

FRED_MONTHLY_PATH  = CLEANED / '02_FRED' / 'fred_monthly_macro_clean.parquet'
MACRO_MONTHLY_PATH = CLEANED / '09_Macro_Daily_Monthly_WRDS' / 'macro_monthly_clean.parquet'

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD FRED MONTHLY AND APPLY PUBLICATION LAG
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: LOAD FRED MONTHLY AND APPLY PUBLICATION LAG")
print("=" * 90)

fred_m = pd.read_parquet(FRED_MONTHLY_PATH)
fred_m['date'] = pd.to_datetime(fred_m['date'])
fred_m_factors = [c for c in fred_m.columns if c != 'date']

print(f"\n  FRED monthly: {len(fred_m):,} rows × {len(fred_m_factors)} factors")
print(f"  Date range: {fred_m['date'].min().date()} → {fred_m['date'].max().date()}")

# Verify monthly frequency
print(f"  Date day-of-month distribution:")
dom = fred_m['date'].dt.day.value_counts().head(5)
print(f"    {dom.to_dict()}")

# ── Apply 2-month publication lag ────────────────────────────────────────────
# FRED monthly dates are reference dates (the month the data describes).
# Most macro indicators are published 1-3 months after the reference month:
#   CPI: ~2 weeks into M+1 (published mid-February for January data)
#   Employment: ~1 week into M+1 (published first Friday of February for January)
#   Industrial Production: ~2 weeks into M+1
#   GDP: ~1 month into M+1 for advance estimate
#   Housing: ~3 weeks into M+1
#
# A blanket 2-month shift is conservative: January's data → available in March.
# This guarantees no look-ahead even for the slowest releases. The cost is
# ~1 month of unnecessary lag on fast-release series — acceptable for a
# dissertation where look-ahead avoidance is paramount.
#
# Implementation: shift the date forward by 2 months. January 2020 data
# gets a date of March 2020, meaning it enters the model in March.

fred_m['date'] = fred_m['date'] + pd.DateOffset(months=2)

# This pushes the last 2 months of data beyond the sample end
# (Nov/Dec 2024 data would be dated Jan/Feb 2025 — outside our sample)
n_before = len(fred_m)
fred_m = fred_m[fred_m['date'] <= '2024-12-31'].reset_index(drop=True)
n_lost = n_before - len(fred_m)

print(f"\n  After 2-month publication lag:")
print(f"  Date range: {fred_m['date'].min().date()} → {fred_m['date'].max().date()}")
print(f"  Rows lost (pushed beyond 2024-12): {n_lost}")
print(f"  Rows remaining: {len(fred_m):,}")

# Add year-month key for merging
fred_m['_ym'] = fred_m['date'].dt.to_period('M')

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: LOAD WRDS MACRO MONTHLY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: LOAD WRDS MACRO MONTHLY")
print("=" * 90)

macro_m = pd.read_parquet(MACRO_MONTHLY_PATH)
macro_m['date'] = pd.to_datetime(macro_m['date'])
macro_m_factors = [c for c in macro_m.columns if c != 'date']

print(f"\n  WRDS macro monthly: {len(macro_m):,} rows × {len(macro_m_factors)} factors")
print(f"  Date range: {macro_m['date'].min().date()} → {macro_m['date'].max().date()}")
print(f"  Columns: {macro_m_factors}")

# ── Separate CPI columns (need 1-month lag) from market-price columns (no lag)
cpi_cols = [c for c in macro_m_factors if c.startswith('cpi')]
market_cols = [c for c in macro_m_factors if c not in cpi_cols]

print(f"\n  Market-price columns (no lag): {len(market_cols)}")
print(f"    {market_cols}")
print(f"  CPI columns (1-month lag): {len(cpi_cols)}")
print(f"    {cpi_cols}")

# ── Apply 1-month lag to CPI columns only ────────────────────────────────────
# CPI for month M is published mid-month M+1 by the BLS.
# Shift CPI forward 1 month: January CPI → available in February.
if cpi_cols:
    cpi_data = macro_m[['date'] + cpi_cols].copy()
    cpi_data['date'] = cpi_data['date'] + pd.DateOffset(months=1)
    cpi_data = cpi_data[cpi_data['date'] <= '2024-12-31'].reset_index(drop=True)
    cpi_data['_ym'] = cpi_data['date'].dt.to_period('M')
    cpi_data = cpi_data.drop(columns='date')
    print(f"\n  CPI after 1-month lag: {len(cpi_data)} rows")

# Market-price columns: no lag
market_data = macro_m[['date'] + market_cols].copy()
market_data['_ym'] = market_data['date'].dt.to_period('M')
market_data = market_data.drop(columns='date')

del macro_m

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: ESTABLISH MONTHLY SPINE AND MERGE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 3: ESTABLISH MONTHLY SPINE AND MERGE")
print("=" * 90)

# Use FRED monthly (post-lag) dates as the spine. These are calendar month-ends
# shifted by 2 months, so the spine represents "when data is available."
# But we also need months that might only exist in WRDS (if FRED was truncated).
# Safest: create a complete monthly calendar from 2004 to 2024.

all_months = pd.date_range('2004-01-31', '2024-12-31', freq='ME')
panel = pd.DataFrame({'date': all_months})
panel['_ym'] = panel['date'].dt.to_period('M')

print(f"\n  Monthly spine: {len(panel)} months")
print(f"  Date range: {panel['date'].min().date()} → {panel['date'].max().date()}")

# ── 3a. Merge FRED monthly (already lagged) ─────────────────────────────────
fred_m_for_merge = fred_m.drop(columns='date')  # keep only _ym + factors
del fred_m

n_before = len(panel)
panel = panel.merge(fred_m_for_merge, on='_ym', how='left')
del fred_m_for_merge
assert len(panel) == n_before, f"Row explosion after FRED merge: {n_before} → {len(panel)}"

n_matched = panel[fred_m_factors[0]].notna().sum()
print(f"\n  + FRED monthly (2-mo lag): {n_matched:,} / {len(panel):,} matched "
      f"({n_matched/len(panel)*100:.1f}%)")

# ── 3b. Merge WRDS market-price columns (no lag) ────────────────────────────
# Check column name conflicts
overlap = set(fred_m_factors) & set(market_cols)
if overlap:
    print(f"  ⚠ Column overlap with FRED: {overlap}")
    market_data = market_data.rename(columns={c: f'wrds_{c}' for c in overlap})
    market_cols = [c if c not in overlap else f'wrds_{c}' for c in market_cols]

n_before = len(panel)
panel = panel.merge(market_data, on='_ym', how='left')
del market_data
assert len(panel) == n_before, f"Row explosion after market merge: {n_before} → {len(panel)}"

n_matched = panel[market_cols[0]].notna().sum()
print(f"  + WRDS market (no lag): {n_matched:,} / {len(panel):,} matched "
      f"({n_matched/len(panel)*100:.1f}%)")

# ── 3c. Merge WRDS CPI columns (1-month lag) ────────────────────────────────
if cpi_cols:
    n_before = len(panel)
    panel = panel.merge(cpi_data, on='_ym', how='left')
    del cpi_data
    assert len(panel) == n_before, f"Row explosion after CPI merge: {n_before} → {len(panel)}"

    n_matched = panel[cpi_cols[0]].notna().sum()
    print(f"  + WRDS CPI (1-mo lag): {n_matched:,} / {len(panel):,} matched "
          f"({n_matched/len(panel)*100:.1f}%)")

# Drop _ym helper
panel = panel.drop(columns='_ym')

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: COLUMN INVENTORY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: COLUMN INVENTORY")
print("=" * 90)

all_factor_cols = [c for c in panel.columns if c != 'date']

fred_in_panel = [c for c in fred_m_factors if c in panel.columns]
market_in_panel = [c for c in market_cols if c in panel.columns]
cpi_in_panel = [c for c in cpi_cols if c in panel.columns]

print(f"\n  Factor breakdown by source:")
print(f"    FRED monthly (2-mo lag):     {len(fred_in_panel):>4d}")
print(f"    WRDS market (no lag):        {len(market_in_panel):>4d}")
print(f"    WRDS CPI (1-mo lag):         {len(cpi_in_panel):>4d}")
print(f"    ────────────────────────────")
print(f"    Total:                       {len(all_factor_cols):>4d}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: VALIDATION")
print("=" * 90)

# ── 5a. No duplicates ───────────────────────────────────────────────────────
n_dupes = panel['date'].duplicated().sum()
print(f"\n  Duplicate dates: {n_dupes}")
assert n_dupes == 0, f"Found {n_dupes} duplicate dates!"

# ── 5b. Shape ───────────────────────────────────────────────────────────────
print(f"\n  Total rows: {len(panel):,}")
print(f"  Total columns: {panel.shape[1]}")
print(f"  Date range: {panel['date'].min().date()} → {panel['date'].max().date()}")
print(f"  Months: {panel['date'].nunique()}")

# ── 5c. Rows per year ───────────────────────────────────────────────────────
print(f"\n  Rows per year:")
rows_year = panel.groupby(panel['date'].dt.year).size()
for y, n in rows_year.items():
    flag = "  ⚠" if n != 12 else ""
    print(f"    {y}: {n:>3d} months{flag}")

# ── 5d. NaN summary by source ───────────────────────────────────────────────
print(f"\n  NaN summary by source:")
for label, cols in [('FRED monthly', fred_in_panel),
                     ('WRDS market', market_in_panel),
                     ('WRDS CPI', cpi_in_panel)]:
    present_cols = [c for c in cols if c in panel.columns]
    if not present_cols:
        continue
    nan_rate = panel[present_cols].isna().mean().mean() * 100
    n_nan = panel[present_cols].isna().sum().sum()
    total = len(panel) * len(present_cols)
    print(f"    {label:<20s} {nan_rate:>5.2f}% avg NaN  "
          f"({n_nan:,} / {total:,} cells)")

# ── 5e. Publication lag verification ────────────────────────────────────────
# The first 2 months should have NaN for FRED (lagged 2 months from start)
# The first 1 month should have NaN for CPI (lagged 1 month)
print(f"\n  Publication lag verification:")
print(f"  (FRED should be NaN for first ~2 months, CPI for first ~1 month)")

for label, cols, expected_nan_months in [
    ('FRED', fred_in_panel[:1], 2),
    ('CPI', cpi_in_panel[:1] if cpi_in_panel else [], 1)
]:
    if not cols:
        continue
    first_valid = panel[panel[cols[0]].notna()]['date'].min()
    print(f"    {label}: first valid date = {first_valid.date()}")

# Also check: WRDS market should start from the very first month
if market_in_panel:
    first_valid_market = panel[panel[market_in_panel[0]].notna()]['date'].min()
    print(f"    WRDS market: first valid date = {first_valid_market.date()} "
          f"(should be Jan 2004)")

# ── 5f. Columns with NaN ────────────────────────────────────────────────────
print(f"\n  Columns with NaN (top 15):")
nan_per_col = panel[all_factor_cols].isna().sum()
nan_cols = nan_per_col[nan_per_col > 0].sort_values(ascending=False)
if len(nan_cols) > 0:
    print(f"\n  {'Column':<35s} {'NaN':>6s}  {'%':>6s}")
    print("  " + "-" * 50)
    for col in nan_cols.head(15).index:
        n = int(nan_cols[col])
        pct = n / len(panel) * 100
        print(f"  {col:<35s} {n:>6,d}  {pct:>5.2f}%")
    if len(nan_cols) > 15:
        print(f"  ... and {len(nan_cols) - 15} more")
else:
    print(f"  ✓ No columns with NaN")

# ── 5g. Date frequency check ────────────────────────────────────────────────
is_month_end = panel['date'].dt.is_month_end
print(f"\n  Dates that are calendar month-end: {is_month_end.sum():,} "
      f"({is_month_end.mean()*100:.1f}%)")

# ── 5h. Sample ──────────────────────────────────────────────────────────────
print(f"\n  Sample (first 5 rows, selected columns):")
sample_cols = ['date', 'b10ret', 'b2ret', 't90ret', 'cpiret',
               'ps_level', 'ps_innov']
sample_cols = [c for c in sample_cols if c in panel.columns]
print(panel[sample_cols].head(5).to_string(index=False))

print(f"\n  Sample (mid-2020):")
mid_2020 = panel[panel['date'] >= '2020-06-30'].head(1)
if len(mid_2020) > 0:
    print(mid_2020[sample_cols].to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 6: SAVE")
print("=" * 90)

panel = panel.sort_values('date').reset_index(drop=True)

out_path = OUT_DIR / 'panel_macro_monthly.parquet'
panel.to_parquet(out_path, index=False, engine='pyarrow')

print(f"\n  ✓ Saved: {out_path}")
print(f"    {len(panel):,} rows × {panel.shape[1]} columns")
print(f"    Key: date")
print(f"    Factors: {len(all_factor_cols)}")
print(f"    Publication lags applied:")
print(f"      FRED monthly: 2-month lag (conservative)")
print(f"      WRDS CPI: 1-month lag")
print(f"      WRDS Treasury/PS: no lag (market prices)")
print(f"    Size: {out_path.stat().st_size / 1e6:.1f} MB")

print("\nPanel D (macro monthly) complete.")

STEP 1: LOAD FRED MONTHLY AND APPLY PUBLICATION LAG

  FRED monthly: 252 rows × 84 factors
  Date range: 2004-01-01 → 2024-12-01
  Date day-of-month distribution:
    {1: 252}

  After 2-month publication lag:
  Date range: 2004-03-01 → 2024-12-01
  Rows lost (pushed beyond 2024-12): 2
  Rows remaining: 250

STEP 2: LOAD WRDS MACRO MONTHLY

  WRDS macro monthly: 252 rows × 22 factors
  Date range: 2004-01-30 → 2024-12-31
  Columns: ['b30ret', 'b30ind', 'b20ret', 'b20ind', 'b10ret', 'b10ind', 'b7ret', 'b7ind', 'b5ret', 'b5ind', 'b2ret', 'b2ind', 'b1ret', 'b1ind', 't90ret', 't90ind', 't30ret', 't30ind', 'cpiret', 'cpiind', 'ps_level', 'ps_innov']

  Market-price columns (no lag): 20
    ['b30ret', 'b30ind', 'b20ret', 'b20ind', 'b10ret', 'b10ind', 'b7ret', 'b7ind', 'b5ret', 'b5ind', 'b2ret', 'b2ind', 'b1ret', 'b1ind', 't90ret', 't90ind', 't30ret', 't30ind', 'ps_level', 'ps_innov']
  CPI columns (1-month lag): 2
    ['cpiret', 'cpiind']

  CPI after 1-month lag: 251 rows

STEP 3: ESTABLISH